# 🚀 AI Subtitle Studio — Google Colab Real AI Acceleration Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/margacis2/ai-subtitle-studio/blob/main/colab_backend.ipynb)

This notebook connects your local **AI Subtitle Studio** web application to **Google Colab's Free GPU (Tesla T4)**.

### Features:
- ⚡ **Real Faster-Whisper ASR** (Tiny, Base, Small, Medium, Large-v3-Turbo)
- 🌐 **Real OPUS-MT Neural Translation** (Japanese, Tamil, Hindi, Telugu, French, German, Spanish, etc.)
- 🔒 **Secure Token Authentication** (Prevents unauthorized requests to your Colab tunnel)
- ☁️ **Cloudflare Quick Tunnel** (Direct HTTPS, no password prompts, no localtunnel IP challenges)
- 🛡️ **Streaming 100 MB Upload Protection** (Guarantees safe upload limit and memory management)
- 🧹 **Automatic Cleanup** (Guaranteed temporary file and VRAM cleanup)

---
### 📖 Quick Setup Guide:
1. **Enable Free GPU**: In the menu bar above, click **Runtime** → **Change runtime type** → select **T4 GPU** → click **Save**.
2. Run **Cell 1** to verify your GPU and system environment.
3. Run **Cell 2** to install verified dependencies & Cloudflare tunnel.
4. Run **Cell 3** to configure and load the backend server.
5. Run **Cell 4** to start the tunnel. Copy the **Engine URL** and **API Token** into **AI Subtitle Studio** (Settings → Colab Addon).

In [ ]:
# @title 1. Environment & Hardware Diagnostics
import sys, os, subprocess, shutil
import torch

print("=" * 60)
print("  AI SUBTITLE STUDIO - GOOGLE COLAB ENVIRONMENT DIAGNOSTICS")
print("=" * 60)
print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Accelerator: [ACTIVE] {device_name} ({vram_gb:.2f} GB VRAM)")
    print(f"CUDA Available : Yes (Compute Capability {torch.cuda.get_device_capability(0)})")
else:
    print("GPU Accelerator: [INACTIVE] Using CPU fallback")
    print("TIP: For maximum speed, click 'Runtime' -> 'Change runtime type' -> 'T4 GPU'!")

ffmpeg_path = shutil.which("ffmpeg")
if ffmpeg_path:
    r = subprocess.run(["ffmpeg", "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    first_line = r.stdout.split("\n")[0] if r.stdout else "Available"
    print(f"FFmpeg         : [OK] {first_line}")
else:
    print("FFmpeg         : Installing via apt-get...")
    os.system("apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1")
    print("FFmpeg         : [OK] Installed")
print("=" * 60)

In [ ]:
# @title 2. Install Pinned Dependencies & Cloudflare Tunnel
import os, sys

print("[*] Installing Faster-Whisper, HuggingFace Transformers, FastAPI & tools...")
!pip install -q -U "faster-whisper>=1.0.0" "transformers>=4.38.0" sentencepiece sacremoses "fastapi>=0.100.0" "uvicorn>=0.23.0" python-multipart pyngrok

# Install cloudflared for direct HTTPS tunnels with NO passwords and NO splash challenges
print("[*] Setting up Cloudflare Tunnel (cloudflared)...\n")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb

print("\n[✓] All dependencies and Cloudflare tunnel installed successfully!")

In [ ]:
# @title 3. Configure & Load AI Backend Engine
import os, sys, tempfile, threading, time, secrets
from typing import List, Optional
import torch
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Depends, Security, Header
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from faster_whisper import WhisperModel
import uvicorn
from transformers import MarianMTModel, MarianTokenizer

# @markdown ### Model & Server Settings
WHISPER_MODEL = "large-v3-turbo"  # @param ["tiny", "base", "small", "medium", "large-v3-turbo"]
API_SECRET_TOKEN = "" # @param {type:"string"}
MAX_UPLOAD_MB = 100 # @param {type:"integer"}

# Generate auto token if user left blank
if not API_SECRET_TOKEN.strip():
    API_SECRET_TOKEN = f"colab-sec-{secrets.token_hex(4)}"

# Determine hardware accelerator
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

print(f"[*] Acceleration Device : {DEVICE.upper()} ({COMPUTE_TYPE})")
print(f"[*] Whisper Model Target: {WHISPER_MODEL}")
print(f"[*] API Secret Token    : {API_SECRET_TOKEN}")
print(f"[*] Max File Upload Size: {MAX_UPLOAD_MB} MB")

# Lazy Model Cache
_whisper_cache = {}
_whisper_lock = threading.Lock()

def get_whisper(model_name: str = WHISPER_MODEL) -> WhisperModel:
    with _whisper_lock:
        if model_name not in _whisper_cache:
            print(f"[+] Loading Faster-Whisper '{model_name}' on {DEVICE} ({COMPUTE_TYPE})...")
            t0 = time.time()
            _whisper_cache[model_name] = WhisperModel(
                model_name,
                device=DEVICE,
                compute_type=COMPUTE_TYPE,
                cpu_threads=4 if DEVICE == "cpu" else 0,
            )
            print(f"[✓] Faster-Whisper '{model_name}' loaded in {time.time() - t0:.2f}s!")
        return _whisper_cache[model_name]

# OPUS-MT Translation Model Map
OPUS_MODEL_MAP = {
    "ja": "Helsinki-NLP/opus-mt-ja-en",
    "ta": "Helsinki-NLP/opus-mt-dra-en",
    "te": "Helsinki-NLP/opus-mt-dra-en",
    "ml": "Helsinki-NLP/opus-mt-dra-en",
    "kn": "Helsinki-NLP/opus-mt-dra-en",
    "hi": "Helsinki-NLP/opus-mt-mul-en",
    "fr": "Helsinki-NLP/opus-mt-fr-en",
    "de": "Helsinki-NLP/opus-mt-de-en",
    "es": "Helsinki-NLP/opus-mt-es-en",
    "zh": "Helsinki-NLP/opus-mt-zh-en",
    "ar": "Helsinki-NLP/opus-mt-ar-en",
    "mul": "Helsinki-NLP/opus-mt-mul-en",
}

_opus_cache = {}
_opus_lock = threading.Lock()

def get_opus_model(src_lang: str):
    lang_key = src_lang.lower().split("-")[0]
    hub_name = OPUS_MODEL_MAP.get(lang_key, OPUS_MODEL_MAP.get("mul"))
    if not hub_name:
        raise ValueError(f"No translation model mapping available for source language: {src_lang}")
    with _opus_lock:
        if hub_name not in _opus_cache:
            print(f"[+] Loading OPUS-MT '{hub_name}' for language '{src_lang}' on {DEVICE}...")
            t0 = time.time()
            tok = MarianTokenizer.from_pretrained(hub_name)
            mdl = MarianMTModel.from_pretrained(hub_name).to(DEVICE)
            mdl.eval()
            _opus_cache[hub_name] = (tok, mdl)
            print(f"[✓] OPUS-MT '{hub_name}' loaded in {time.time() - t0:.2f}s!")
        return _opus_cache[hub_name]

def translate_texts_with_opus(texts: List[str], src_lang: str, tgt_lang: str = "en") -> List[str]:
    if not texts:
        return []
    if src_lang == tgt_lang or (src_lang == "en" and tgt_lang == "en"):
        return texts
    tok, mdl = get_opus_model(src_lang)
    batch_size = 16
    results = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i : i + batch_size]
        clean_chunk = [t if t.strip() else "." for t in chunk]
        inp = tok(clean_chunk, return_tensors="pt", padding=True, truncation=True, max_length=512)
        if DEVICE == "cuda":
            inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.no_grad():
            out = mdl.generate(**inp, max_length=512)
        decoded = [tok.decode(t, skip_special_tokens=True).strip() for t in out]
        for original, trans in zip(chunk, decoded):
            results.append("" if not original.strip() else trans)
    return results

# FastAPI Application Definition
app = FastAPI(title="AI Subtitle Studio - Colab GPU Engine", version="2.0.1")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_token(authorization: Optional[str] = Header(None)):
    if not API_SECRET_TOKEN:
        return True
    if not authorization:
        raise HTTPException(status_code=401, detail="Missing Authorization Bearer token header")
    parts = authorization.split()
    if len(parts) != 2 or parts[0].lower() != "bearer":
        raise HTTPException(status_code=401, detail="Invalid Authorization header format. Expected 'Bearer <token>'")
    token = parts[1]
    if token != API_SECRET_TOKEN:
        raise HTTPException(status_code=403, detail="Invalid API Secret Token")
    return True

@app.get("/health")
async def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {
        "status": "online",
        "whisper_model": WHISPER_MODEL,
        "device": f"Colab {gpu_name} ({DEVICE.upper()})",
        "compute_type": COMPUTE_TYPE,
        "max_upload_mb": MAX_UPLOAD_MB,
        "has_token": bool(API_SECRET_TOKEN),
    }

class TranslateRequest(BaseModel):
    texts: List[str]
    source_lang: str = "ja"
    target_lang: str = "en"

@app.post("/translate")
async def translate_endpoint(req: TranslateRequest, authorized: bool = Depends(verify_token)):
    try:
        translated = translate_texts_with_opus(req.texts, req.source_lang, req.target_lang)
        return {
            "source_lang": req.source_lang,
            "target_lang": req.target_lang,
            "translations": translated,
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/transcribe")
async def transcribe_audio(
    file: UploadFile = File(...),
    source_lang: str = Form("auto"),
    target_lang: str = Form("en"),
    authorized: bool = Depends(verify_token)
):
    max_bytes = MAX_UPLOAD_MB * 1024 * 1024
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
        bytes_written = 0
        while chunk := await file.read(1024 * 1024):  # 1MB stream chunks
            bytes_written += len(chunk)
            if bytes_written > max_bytes:
                tmp.close()
                if os.path.exists(tmp.name):
                    os.remove(tmp.name)
                raise HTTPException(
                    status_code=413,
                    detail=f"Uploaded audio file exceeds {MAX_UPLOAD_MB} MB limit ({bytes_written / (1024*1024):.1f} MB received). Please optimize audio below 100 MB."
                )
            tmp.write(chunk)
        tmp_path = tmp.name

    try:
        model = get_whisper(WHISPER_MODEL)
        lang = None if (not source_lang or source_lang == "auto") else source_lang

        print(f"[*] Transcribing audio ({bytes_written / (1024*1024):.2f} MB, lang={lang or 'auto'})...")
        t0 = time.time()
        segments_gen, info = model.transcribe(
            tmp_path,
            language=lang,
            task="transcribe",
            beam_size=5,
            best_of=5,
            temperature=[0.0, 0.2, 0.4],
            compression_ratio_threshold=2.2,
            log_prob_threshold=-1.0,
            no_speech_threshold=0.6,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=400),
            condition_on_previous_text=False
        )

        raw_segments = list(segments_gen)
        detected_lang = info.language
        original_texts = [s.text.strip() for s in raw_segments]

        # OPUS-MT Translation if requested
        effective_src = lang or detected_lang or "ja"
        if target_lang != "original" and target_lang != effective_src and original_texts:
            try:
                translated_texts = translate_texts_with_opus(original_texts, effective_src, target_lang)
            except Exception as e:
                print(f"[!] Translation note: {e}. Falling back to original transcription.")
                translated_texts = original_texts
        else:
            translated_texts = original_texts

        results = []
        for s, orig_text, trans_text in zip(raw_segments, original_texts, translated_texts):
            if orig_text:
                results.append({
                    "startMs": int(s.start * 1000),
                    "endMs": int(s.end * 1000),
                    "original": orig_text,
                    "translation": trans_text,
                    "text": trans_text if target_lang != "original" else orig_text,
                    "confidence": int(round((1.0 - getattr(s, "no_speech_prob", 0.0)) * 100)),
                    "lang": detected_lang
                })

        print(f"[✓] Transcribed {len(results)} segments in {time.time() - t0:.2f}s (language: {detected_lang})")
        return {"language": detected_lang, "segments": results}
    finally:
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except Exception:
                pass

In [ ]:
# @title 4. Start Server & Expose via Cloudflare Tunnel
import subprocess, time, re, sys, threading

# 1. Start Uvicorn in background thread
def run_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8001, log_level="warning")

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)

# 2. Launch cloudflared tunnel
print("[*] Spawning Cloudflare Quick Tunnel (direct HTTPS, no password prompts)...\n")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

# Read lines from stderr (cloudflared logs to stderr)
start_time = time.time()
while time.time() - start_time < 35:
    line = tunnel_proc.stderr.readline()
    if not line:
        continue
    match = pattern.search(line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    print("[!] Cloudflare quick tunnel did not establish in 35s. Launching Localtunnel fallback...")
    import urllib.request
    ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print(f"Localtunnel IP Password: {ip}")
    !npx localtunnel --port 8001
else:
    print("\n" + "=" * 65)
    print("  🚀 AI SUBTITLE STUDIO - GOOGLE COLAB GPU BACKEND ONLINE!")
    print("=" * 65)
    print(f"  🔗 Engine URL    : {public_url}")
    print(f"  🔑 API Token     : {API_SECRET_TOKEN}")
    print(f"  ⚡ Model Target  : Faster-Whisper {WHISPER_MODEL} ({DEVICE.upper()})")
    print("=" * 65)
    print("\n  👉 Next Steps in your AI Subtitle Studio App:")
    print("  1. Open AI Subtitle Studio in your browser.")
    print("  2. Click 'Settings' (top right corner).")
    print("  3. Under 'Google Colab Remote GPU', enter:")
    print(f"     • Tunnel / Engine URL: {public_url}")
    print(f"     • API Secret Token   : {API_SECRET_TOKEN}")
    print("  4. Click 'Save & Verify Connection'.")
    print("  5. Enjoy unlimited, ultra-fast cloud GPU transcription and translation!\n")
    print("=" * 65)

    # Keep process streaming in foreground so Colab cell stays active
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n[!] Stopping server and tunnel...")
        tunnel_proc.terminate()

In [ ]:
# @title 5. [Optional] Alternative: Ngrok Tunnel (If Cloudflare is blocked in your network)
# @markdown Enter your free Ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "" # @param {type:"string"}

if NGROK_AUTHTOKEN.strip():
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    ngrok_tunnel = ngrok.connect(8001)
    print("=" * 60)
    print(f"  🔗 Ngrok Public URL: {ngrok_tunnel.public_url}")
    print(f"  🔑 API Token       : {API_SECRET_TOKEN}")
    print("=" * 60)
else:
    print("To use Ngrok, please enter your NGROK_AUTHTOKEN above and run this cell.")

In [ ]:
# @title 6. End-to-End Self-Test (Verify ASR & Translation Directly)
# @markdown Run this cell to verify speech recognition & translation directly inside Colab!
import wave, math, struct

# Generate a clean 3-second test WAV audio tone
test_wav = "colab_self_test.wav"
with wave.open(test_wav, "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(16000)
    # Generate 16kHz sine wave speech-band tone (440 Hz)
    samples = [int(16000 * math.sin(2 * math.pi * 440 * i / 16000)) for i in range(16000 * 3)]
    wf.writeframes(struct.pack(f"<{len(samples)}h", *samples))

print("[*] Testing Faster-Whisper model loading & transcription...")
model = get_whisper(WHISPER_MODEL)
seg_gen, info = model.transcribe(test_wav, vad_filter=False)
segs = list(seg_gen)
print(f"[✓] Whisper ASR self-test completed! Detected lang: {info.language}")

print("[*] Testing OPUS-MT translation (Japanese -> English)...\n")
sample_text = ["こんにちは、世界！", "AI字幕スタジオへようこそ。"]
res = translate_texts_with_opus(sample_text, "ja", "en")
for orig, trans in zip(sample_text, res):
    print(f"    Original   : {orig}")
    print(f"    Translation: {trans}")

print("\n[✓] ALL TESTS PASSED! Colab AI Engine is 100% verified and operational.")